# SkyPortal - Fetch candidates for a group

This notebook uses the SkyPortal API to retrieve candidates associated with a given group.

In [14]:
import os
import pandas as pd
from dotenv import load_dotenv

from utils.skyportal_api import SkyPortal

load_dotenv()

SKYPORTAL_URL = "https://orcusgate.org/"
SKYPORTAL_TOKEN = os.getenv("ORCUS_TOKEN")

if not SKYPORTAL_TOKEN:
    raise ValueError("SKYPORTAL_TOKEN is missing. Set it in the .env file.")

skyPortal = SkyPortal(SKYPORTAL_URL, SKYPORTAL_TOKEN)
print(f"Connected to {SKYPORTAL_URL}")

Connected to https://orcusgate.org/


## 1. List accessible groups

In [15]:
groups = skyPortal.api("GET", "/api/groups")["user_groups"]

groups_df = pd.DataFrame([{"id": g["id"], "name": g["name"]} for g in groups])
groups_df

,id,name
0,12,LIONS ZTF
1,2,Sitewide Group
2,14,ssharmac
3,19,Superphot LSST
4,13,Superphot ZTF


## 2. Select the group and fetch candidates

In [16]:
GROUP_ID = 13
LAST_N_DAYS = 4  # Number of days to look back (1 = last night)
START_DATE = None # Optional: specific start date (overrides LAST_N_DAYS) e.g. "2026-03-04T18:00:00"

In [17]:
from datetime import datetime, timedelta, timezone

start_date = START_DATE or (datetime.now(timezone.utc) - timedelta(days=LAST_N_DAYS)).strftime("%Y-%m-%dT%H:%M:%S")

candidates = skyPortal.fetch_all_pages(
    "/api/candidates",
    payload={"groupIDs": str(GROUP_ID), "startDate": start_date},
    item_key="candidates",
)
print(f"{len(candidates)} candidates passed since {start_date} UTC")

42 candidates passed since 2026-03-01T12:12:43 UTC


In [18]:
from datetime import datetime, timedelta, timezone

start_date = START_DATE or (datetime.now(timezone.utc) - timedelta(days=LAST_N_DAYS)).strftime("%Y-%m-%dT%H:%M:%S")

candidates = skyPortal.fetch_all_pages(
    "/api/candidates",
    payload={"groupIDs": str(GROUP_ID), "startDate": start_date},
    item_key="candidates",
)
print(f"{len(candidates)} candidates passed since {start_date} UTC")

42 candidates passed since 2026-03-01T12:12:45 UTC


## 3. Display results

In [19]:
df = pd.DataFrame([
    {
        "id": c["id"],
        "ra": c.get("ra"),
        "dec": c.get("dec"),
        "redshift": c.get("redshift"),
        "last_detected_at": c.get("last_detected_at"),
        "classifications": ", ".join(
            cl["classification"] for cl in c.get("classifications", [])
        ),
    }
    for c in candidates
])

print(f"{len(df)} candidates for group {GROUP_ID}")
df.head(20)

42 candidates for group 13


,id,ra,dec,redshift,last_detected_at,classifications
0,ZTF26aahyyny,120.848618,8.290481,None,2026-03-03T07:17:32.000637,
1,ZTF26aahpthq,132.348036,18.525050,None,2026-03-03T07:05:47.002570,
2,ZTF26aahfrag,138.980906,10.135896,None,2026-03-03T07:03:43.001271,
3,ZTF26aahaesb,132.356133,29.510539,None,2026-03-03T06:58:05.998070,
4,ZTF26aahajej,123.776604,17.822087,None,2026-03-03T05:51:16.001273,
5,ZTF26aahauhi,114.896892,27.636411,None,2026-03-03T05:58:51.000978,
6,ZTF26aabpbkj,130.820612,3.689420,None,2026-03-03T05:52:38.003524,
7,ZTF26aabjpnx,144.567295,26.688913,None,2026-03-03T04:08:22.004144,
8,ZTF26aaclbkw,72.821795,0.427953,None,2026-03-03T03:34:09.001922,
9,ZTF25acgklik,72.001587,-16.661684,None,2026-03-03T03:04:43.003213,


## LSST objects

In [20]:
GROUP_ID = 19
LAST_N_DAYS = 6  # Number of days to look back (1 = last night)
START_DATE = None # Optional: specific start date (overrides LAST_N_DAYS) e.g. "2026-03-04T18:00:00"

In [21]:
from datetime import datetime, timedelta, timezone

start_date = START_DATE or (datetime.now(timezone.utc) - timedelta(days=LAST_N_DAYS)).strftime("%Y-%m-%dT%H:%M:%S")

candidates = skyPortal.fetch_all_pages(
    "/api/candidates",
    payload={"groupIDs": str(GROUP_ID), "startDate": start_date},
    item_key="candidates",
)
print(f"{len(candidates)} candidates passed since {start_date} UTC")

36 candidates passed since 2026-02-27T12:12:46 UTC


In [22]:
df = pd.DataFrame([
    {
        "id": c["id"],
        "ra": c.get("ra"),
        "dec": c.get("dec"),
        "redshift": c.get("redshift"),
        "last_detected_at": c.get("last_detected_at"),
        **(c["annotations"][0].get("data", {}) if c.get("annotations") else {}),
    }
    for c in candidates
])

print(f"{len(df)} candidates for group {GROUP_ID}")
df.head(20)

36 candidates for group 19


,id,ra,dec,redshift,last_detected_at,SN II,SN Ia,SLSN-I,SN IIn,SN Ibc
0,170032884263419978,59.819416,-49.283959,None,2026-03-03T02:50:49.272953,NaN,NaN,NaN,NaN,NaN
1,313871013579325450,57.796707,-48.198582,None,2026-03-02T01:48:04.959273,0.000,0.000,1.000,0.000,0.000
2,170046083740205236,56.875607,-49.855863,None,2026-03-02T00:43:22.282204,0.037,0.306,0.575,0.062,0.020
3,170028486178635819,63.107529,-49.494713,None,2026-03-01T01:55:38.141888,0.114,0.000,0.000,0.000,0.886
4,170028488414724130,60.738398,-47.528347,None,2026-03-01T01:52:17.932916,0.041,0.000,0.000,0.004,0.955
5,313681913501974541,63.493126,-47.209148,None,2026-03-01T01:52:17.932916,0.000,0.000,1.000,0.000,0.000
6,313690717425238035,61.068281,-46.907672,None,2026-03-01T01:52:17.932916,NaN,NaN,NaN,NaN,NaN
7,313677516844302755,60.628203,-47.427895,None,2026-03-01T01:52:17.932916,1.000,0.000,0.000,0.000,0.000
8,170046086089015366,61.637261,-46.522112,None,2026-03-01T01:52:17.932916,0.541,0.281,0.005,0.032,0.141
9,170028485719359556,61.429067,-46.546131,None,2026-03-01T01:52:17.932916,0.000,0.621,0.379,0.000,0.000
